Importando As bibliotecas

In [2]:
import pandas as pd
import matplotlib as plot
import numpy as np
import matplotlib.pyplot as plt

Criando as Tabelas

In [3]:
pacientes = pd.read_csv("mimic-code/mimic-iv-demo/2.2/hosp/patients.csv", sep=',')
admissoes = pd.read_csv("mimic-code/mimic-iv-demo/2.2/hosp/admissions.csv", sep=',')
eventos_hospital = pd.read_csv("mimic-code/mimic-iv-demo/2.2/hosp/emar.csv", sep=',')


Dados dos Pacientes, como Idade, ultima data de entrada e ultima alta e data da morte, caso tenha

In [ ]:
pt = pacientes[["subject_id","anchor_age","dod"]]
pacientes_completo = pd.merge(pt,admissoes,on="subject_id",how="inner")

pacientes_completo = pacientes_completo[["subject_id","anchor_age","hadm_id","admittime","dischtime","dod"]]

pacientes_completo #todas as consultas dos pacientes

pacientes_ultimo = pacientes_completo.sort_values(by=["subject_id","dischtime"],ascending=True)

pacientes_ultimo = pacientes_ultimo.groupby("subject_id").last().reset_index()

print(pacientes_ultimo.to_string()) #ultimas consultas dos pacientes


In [90]:
import pandas as pd

# Carregar os dados (substitua pelos caminhos corretos dos arquivos CSV)
diagnoses_icd = pd.read_csv('mimic-code/mimic-iv-demo/2.2/hosp/diagnoses_icd.csv')
admissions = pd.read_csv('mimic-code/mimic-iv-demo/2.2/hosp/admissions.csv')

# Criar colunas icd9_code e icd10_code
diagnoses_icd['icd9_code'] = diagnoses_icd.apply(
    lambda row: row['icd_code'] if row['icd_version'] == 9 else None, axis=1
)
diagnoses_icd['icd10_code'] = diagnoses_icd.apply(
    lambda row: row['icd_code'] if row['icd_version'] == 10 else None, axis=1
)

# Função para verificar condições com base nos códigos ICD-9 e ICD-10
def check_condition(df, icd9_codes, icd10_codes, col_name):
    df[col_name] = df.apply(
        lambda row: 1 if (
            (row['icd9_code'] and any(row['icd9_code'].startswith(code) for code in icd9_codes)
            or (row['icd10_code'] and any(row['icd10_code'].startswith(code) for code in icd10_codes)
        ))) else 0, axis=1
    )
    return df

# Juntar as tabelas diagnoses_icd e admissions
merged_df = pd.merge(admissions, diagnoses_icd, on='hadm_id', how='left')

# Listas de códigos ICD-9 e ICD-10 para cada comorbidade
conditions = {
    'myocardial_infarct': {
        'icd9': ['410', '412'],
        'icd10': ['I21', 'I22', 'I252']
    },
    'congestive_heart_failure': {
        'icd9': ['428', '39891', '40201', '40211', '40291', '40401', '40403',
                 '40411', '40413', '40491', '40493', '4254', '4255', '4256',
                 '4257', '4258', '4259'],
        'icd10': ['I43', 'I50', 'I099', 'I110', 'I130', 'I132', 'I255', 'I420',
                  'I425', 'I426', 'I427', 'I428', 'I429', 'P290']
    },
    'Peripheral vascular disease':{
        'icd9' : ['440', '441','0930', '4373', '4471', '5571', '5579', 'V434','4431','4432','4433',
                  '4434','4435','4436','4437','4438','4439',],
        'icd10': ['I70', 'I71','I731', 'I738', 'I739', 'I771', 'I790', 'I792', 'K551', 'K558', 'K559', 'Z958', 'Z959']
    },
    'Cerebrovascular disease' :{
        'icd9':['430','431','432','433','434','435','436','437','438','439','36234','G45', 'G46'],
        'icd10': ['I60','I61','I62','I63','I64','I65','I66','I67','I68','I69','H340']
    },
    'Dementia':{
        'icd9':['290','2941', '3312'],
        'icd10': ['F051', 'G311','F00', 'F01', 'F02', 'F03', 'G30']
    },
    'Chronic pulmonary disease':{
        'icd9':['490','491','492','493','494','495','496','497','498','499','500','501','502','503','504','505','4168'
                , '4169', '5064', '5081', '5088'],
        'icd10':['J40','J41','J42','J43','J44','J45','J46','J47','J60','J61','J62','J63','J64','J65','J66','J67','I278',
                  'I279', 'J684', 'J701', 'J703']
    },
    'Rheumatic disease':{
        'icd9': ['725','4465', '7100', '7101', '7102', '7103', '7104', '7140', '7141', '7142', '7148'],
        'icd10': ['M05', 'M06', 'M32', 'M33', 'M34','M315', 'M351', 'M353', 'M360']
    },
    'Peptic ulcer disease': {
        'icd9': ['531', '532', '533', '534'],
        'icd10': ['K25', 'K26', 'K27', 'K28']
    },
    'Mild liver disease' :{
        'icd9': ['570', '571','0706', '0709', '5733', '5734', '5738', '5739', 'V427','07022', '07023', '07032', '07033', '07044', '07054'],
        'icd10': ['B18', 'K73', 'K74','K700', 'K701', 'K702', 'K703', 'K709', 'K713'
                  , 'K714', 'K715', 'K717', 'K760', 'K762'
                  , 'K763', 'K764', 'K768', 'K769', 'Z944']
    },
    'Diabetes without chronic complication' : {
        'icd9': ['2500', '2501', '2502', '2503', '2508', '2509'],
        'icd10': ['E100', 'E101', 'E106', 'E108', 'E109', 'E110', 'E111' , 'E116' , 'E118' , 'E119' , 'E120' , 'E121' , 'E126' , 'E128' ,
                   'E129' , 'E130' , 'E131' , 'E136' , 'E138' , 'E139' , 'E140' , 'E141', 'E146', 'E148', 'E149']
    },
    'Diabetes with chronic complication': {
        'icd9': ['2504', '2505', '2506', '2507'],
        'icd10' :['E102', 'E103', 'E104', 'E105', 'E107', 'E112', 'E113' , 'E114' , 'E115' , 'E117' , 'E122' , 'E123' , 'E124' , 'E125' ,
                   'E127' , 'E132' , 'E133' , 'E134' , 'E135' , 'E137' , 'E142' , 'E143', 'E144', 'E145', 'E147']
    },
    'Hemiplegia or paraplegia': {
        'icd9': ['342', '343','3341', '3440', '3441', '3442' , '3443', '3444', '3445', '3446', '3449'],
        'icd10': ['G81', 'G82','G041', 'G114', 'G801', 'G802', 'G830' , 'G831' , 'G832' , 'G833' , 'G834' , 'G839']
    },
    'Renal disease' :{
        'icd9': ['582', '585', '586', 'V56','5880', 'V420', 'V451','5830','5831','5832','5833','5834','5835','5836','5837','40301' , '40311' ,
                  '40391' , '40402' , '40403' , '40412' , '40413' , '40492' , '40493'],
        'icd10': ['N18', 'N19','I120', 'I131', 'N032', 'N033', 'N034' , 'N035' , 'N036' , 'N037' , 'N052' , 'N053' , 'N054' , 'N055' , 'N056' ,
                   'N057' , 'N250' , 'Z490' , 'Z491' , 'Z492' , 'Z940' , 'Z992']
    },
    'AIDS/HIV' :{
        'icd9': ['042', '043', '044'],
        'icd10':['B20', 'B21', 'B22', 'B24']
    },
    'Hypertension' : {
        'icd9':  ['4019', '5723'],
        'icd10': ['I2720', 'I272', 'K766', 'I10']
    },
    'Hypoglycemia':{
        'icd9': ['2511'],
        'icd10': ['E11649', 'E162']
    },
    'Hyperglycemia':{
        'icd9': [],
        'icd10': ['E0865', 'E1165']
    }
    ,
    'tabacco':{
        'icd9': ['3051', 'V1582'],
        'icd10': ['Z720']
    }
}

# Aplicar as condições para cada comorbidade
for condition, codes in conditions.items():
    merged_df = check_condition(merged_df, codes['icd9'], codes['icd10'], condition)

# Agrupar por hadm_id e calcular o máximo para cada comorbidade
comorbidities = merged_df.groupby('subject_id_x').agg({
    'myocardial_infarct': 'sum',
    'congestive_heart_failure': 'sum',
    'Peripheral vascular disease': 'sum',
    'Cerebrovascular disease': 'sum',
    'Dementia':'sum',
    'Chronic pulmonary disease': 'sum',
    'Diabetes with chronic complication': 'sum',
    'Diabetes without chronic complication': 'sum',
    'Rheumatic disease': 'sum',
    'Peptic ulcer disease': 'sum',
    'Mild liver disease': 'sum',
    'Hemiplegia or paraplegia': 'sum',
    'AIDS/HIV': 'sum',
    'Renal disease': 'sum',
    'Hypertension': 'sum',
    'Hypoglycemia': 'sum',
    'Hyperglycemia': 'sum',
    'tabacco': 'sum',
    # Adicione as outras colunas de comorbidades aqui...
}).reset_index()

# Exibir o resultado
#print(comorbidities)
comorbidities


#print(comorbidities["subject_id_x"].nunique())

,subject_id_x,myocardial_infarct,congestive_heart_failure,Peripheral vascular disease,Cerebrovascular disease,Dementia,Chronic pulmonary disease,Diabetes with chronic complication,Diabetes without chronic complication,Rheumatic disease,Peptic ulcer disease,Mild liver disease,Hemiplegia or paraplegia,AIDS/HIV,Renal disease,Hypertension,Hypoglycemia,Hyperglycemia,tabacco
0,10000032,0,0,0,0,0,4,0,0,0,0,6,0,0,0,1,0,0,4
1,10001217,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,2
2,10001725,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
3,10002428,0,2,0,0,1,1,0,0,5,0,0,0,0,0,7,0,0,0
4,10002495,2,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,10038999,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
96,10039708,1,1,0,0,0,6,0,0,0,1,12,0,0,16,2,0,0,2
97,10039831,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
98,10039997,0,0,1,7,0,0,0,0,0,0,0,1,0,1,2,0,0,1


In [56]:


# Carregar os arquivos CSV (substitua pelos caminhos corretos)
diagnoses_icd = pd.read_csv('mimic-code/mimic-iv-demo/2.2/hosp/diagnoses_icd.csv')
d_icd_diagnoses = pd.read_csv('mimic-code/mimic-iv-demo/2.2/hosp/d_icd_diagnoses.csv')

# Juntar as tabelas diagnoses_icd e d_icd_diagnoses
merged_df = pd.merge(diagnoses_icd, d_icd_diagnoses, on=['icd_code', 'icd_version'], how='left')

# Criar listas para armazenar os códigos ICD-9 e ICD-10 relacionados à "hypertension"
icd9_codes = set()
icd10_codes = set()

# Filtrar os códigos cujas labels contenham "hypertension" (ignorando maiúsculas/minúsculas)
for index, row in merged_df.iterrows():
    if row['long_title'].lower().find('tobacco') != -1:  # Verifica se "hypertension" está na label
        if row['icd_version'] == 9:
            icd9_codes.add(row['icd_code'])  # Adiciona à lista de ICD-9
        elif row['icd_version'] == 10:
            icd10_codes.add(row['icd_code'])  # Adiciona à lista de ICD-10

# Converter os conjuntos de volta para listas (se necessário)
icd9_codes = list(icd9_codes)
icd10_codes = list(icd10_codes)

# Exibir os resultados
print("Códigos ICD-9 relacionados à 'hypertension':", icd9_codes)
print("Códigos ICD-10 relacionados à 'hypertension':", icd10_codes)

Códigos ICD-9 relacionados à 'hypertension': ['3051', 'V1582']
Códigos ICD-10 relacionados à 'hypertension': ['Z720']
